### DDQN Agent Evaluation on Test Data

In [ ]:
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.append('..')
from environment.trading_env import TradingEnv
from experts.ddqn_expert import DDQNAgent

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
ASSET = "EURUSD"  # change to your asset
CONFIG_PATH = "../configs/experts.yaml"
TEST_DATA_PATH = f"../data/processed/{ASSET}/train.csv"
MODEL_PATH = f"../checkpoints/{ASSET}/no_diversity/DDQN/ddqn_expert.pt"

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)
ddqn_cfg = cfg["ddqn"]

In [ ]:
# Load test data
test_df = pd.read_csv(TEST_DATA_PATH, index_col=0, parse_dates=True)
print(f"Test data shape: {test_df.shape}")

# Load checkpoint
ckpt = torch.load(MODEL_PATH, map_location='cpu')
print(f"Checkpoint keys: {list(ckpt.keys())}")

# Get expected state dimension from Q-network
q_state = ckpt["q_network"]
# DDQN's DuelingQNetwork has a 'trunk' attribute (Sequential)
# The first linear layer key is usually 'trunk.0.weight'
first_layer_key = None
for key in q_state.keys():
    if "weight" in key and ("trunk" in key or "0" in key):
        first_layer_key = key
        break
if first_layer_key is None:
    raise KeyError("Could not find first layer in Q-network state dict.")

expected_input_dim = q_state[first_layer_key].shape[1]
expected_features = expected_input_dim - 6   # environment adds 6 state features
print(f"Checkpoint expects {expected_features} features (input dim {expected_input_dim}).")

# Use config features if they match, otherwise adapt
config_features = ddqn_cfg.get("features", [])
if len(config_features) == expected_features:
    features = config_features
else:
    print(f"Config has {len(config_features)} features, but checkpoint expects {expected_features}.")
    # Adapt: try to pick technical indicators from test_df columns
    exclude = ["open", "high", "low", "close", "volume", "log_volume", "returns", "return_5"]
    candidates = [c for c in test_df.columns if c not in exclude]
    if len(candidates) >= expected_features:
        features = candidates[:expected_features]
    else:
        features = list(test_df.columns[:expected_features])
    print("Using adapted features:", features)

In [ ]:
asset_type = "crypto" if "USDT" in ASSET else "forex"
window_size = cfg.get("window_size", 20)

env_params = {
    "df": test_df,
    "feature_cols": features,
    "expert_id": 3,                      # DDQN = expert 3
    "asset_type": asset_type,
    "contract_size": 1.0 if asset_type == "crypto" else 100000.0,
    "commission_rate": 0.0005 if asset_type == "crypto" else 1.5/100000.0,
    "spread_rate": 0.0002 if asset_type == "crypto" else 0.00005,
    "slippage_rate": 0.0001 if asset_type == "crypto" else 0.00001,
    "leverage": 5 if asset_type == "crypto" else 30,
    "maintenance_margin_ratio": 0.5,
    "initial_balance": 10000.0,
    "window_size": window_size,
    "reward_scale": 1.0,
    "clip_reward": (-10.0, 10.0),
    "max_position_fraction": 0.95,
    "continuous_actions": False,        # DDQN uses discrete actions (3)
}
env = TradingEnv(**env_params)

# Instantiate DDQN agent with hyperparameters from config
state_dim = len(features) + 6
agent = DDQNAgent(
    state_dim=state_dim,
    action_dim=3,                       # Discrete: hold, long, short
    epsilon_start=0,
    epsilon_end=0,
    dueling=ddqn_cfg.get("dueling", True),
    hidden_dim=ddqn_cfg.get("hidden_dim", 256),
    num_layers=ddqn_cfg.get("num_layers", 2),
    activation=ddqn_cfg.get("activation", "relu"),
    log_dir=None,
    device="cpu",
)

# Load weights
agent.q_network.load_state_dict(ckpt["q_network"])
agent.target_network.load_state_dict(ckpt["target_network"])
if "total_steps" in ckpt:
    agent._total_steps = ckpt["total_steps"]

print("DDQN agent loaded.")

In [ ]:
obs, _ = env.reset()
done = False
equity_curve = [env.initial_balance]   # track equity

while not done:
    # DDQN's act returns integer action
    action = agent.act(obs)
    obs, reward, terminated, truncated, info = env.step(action)
    equity_curve.append(info.get('equity', env.initial_balance))
    done = terminated or truncated

equity_curve = np.array(equity_curve)
cum_return = (equity_curve - equity_curve[0]) / equity_curve[0]

print(f"Final equity: {equity_curve[-1]:.2f}")
print(f"Final cumulative return: {cum_return[-1]:.4f} ({cum_return[-1]*100:.2f}%)")

In [ ]:
# Daily returns (percentage change of equity)
daily_returns = np.diff(equity_curve) / equity_curve[:-1]
total_ret = cum_return[-1]
n = len(daily_returns)
ann_ret = (1 + total_ret) ** (252 / n) - 1 if n > 0 else 0.0
ann_vol = np.std(daily_returns) * np.sqrt(252)
sharpe = ann_ret / ann_vol if ann_vol != 0 else 0.0

# Drawdown (based on equity)
peak = np.maximum.accumulate(equity_curve)
drawdown = (peak - equity_curve) / peak
max_dd = np.max(drawdown)

win_rate = np.mean(daily_returns > 0)

print("\n--- DDQN Performance ---")
print(f"Total Return:       {total_ret:.2%}")
print(f"Annualized Return:  {ann_ret:.2%}")
print(f"Annualized Vol:     {ann_vol:.2%}")
print(f"Sharpe Ratio:       {sharpe:.2f}")
print(f"Max Drawdown:       {max_dd:.2%}")
print(f"Win Rate:           {win_rate:.2%}")

In [ ]:
# Equity curve (cumulative return)
plt.figure(figsize=(14,6))
plt.plot(cum_return, label='DDQN', linewidth=2)
plt.title("DDQN Equity Curve (Cumulative Return)")
plt.xlabel("Step")
plt.ylabel("Cumulative Return")
plt.grid(True)
plt.legend()
plt.show()

# Drawdown
plt.figure(figsize=(14,3))
plt.fill_between(range(len(drawdown)), 0, drawdown, color='red', alpha=0.4)
plt.plot(drawdown, color='darkred', linewidth=0.5)
plt.title("Drawdown")
plt.xlabel("Step")
plt.ylabel("Drawdown")
plt.grid(True)
plt.show()